# Cargando datos de Python a SQL

In [23]:
#Obteniendo de 1_Limpieza_data.ipynb para utilizar variables
%run "D:\Proyectos\PYTHON\Project-Superstore-Retail-Analysis\1_Limpieza_data.ipynb"

(9994, 21)
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
<class 'pandas.core.frame.DataFrame'>


In [24]:
"""
Versión con SQLAlchemy
"""

import pandas as pd
from sqlalchemy import create_engine

server   = r'LAPTOP-LK9MMHNU\SQLEXPRESS'
database = 'SuperstoreRetail'
driver   = 'ODBC Driver 17 for SQL Server'

conn_str = f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver={driver}'
engine   = create_engine(conn_str)

# --- CUSTOMERS ---
df_customers = df_customers.rename(columns={
    'Customer ID'   : 'customer_id',
    'Customer Name' : 'customer_name',
    'Segment'       : 'segment',
    'Country'       : 'country',
    'City'          : 'city',
    'State'         : 'state',
    'Postal Code'   : 'postal_code',
    'Region'        : 'region'
})

# --- PRODUCTS ---
df_products = df_products.rename(columns={
    'Product ID'   : 'product_id',
    'Product Name' : 'product_name',
    'Category'     : 'category',
    'Sub-Category' : 'sub_category'
})

# --- SALES ---
df_sales = df_sales.rename(columns={
    'Row ID'      : 'row_id',
    'Order ID'    : 'order_id',
    'Order Date'  : 'order_date',
    'Ship Date'   : 'ship_date',
    'Ship Mode'   : 'ship_mode',
    'Customer ID' : 'customer_id',
    'Product ID'  : 'product_id',
    'Sales'       : 'sales',
    'Quantity'    : 'quantity',
    'Discount'    : 'discount',
    'Profit'      : 'profit'
})

# --- INSERTAR EN SQL SERVER ---
df_customers.to_sql('Customers', engine, if_exists='append', index=False)
df_products.to_sql('Products',   engine, if_exists='append', index=False)
df_sales.to_sql('Sales',         engine, if_exists='append', index=False)

print(f"Customers : {len(df_customers)} filas insertadas")
print(f"Products  : {len(df_products)} filas insertadas")
print(f"Sales     : {len(df_sales)} filas insertadas")

Customers : 793 filas insertadas
Products  : 1862 filas insertadas
Sales     : 9994 filas insertadas


In [25]:
import pandas as pd
from sqlalchemy import create_engine

server   = r'LAPTOP-LK9MMHNU\SQLEXPRESS'
database = 'SuperstoreRetail'
driver   = 'ODBC Driver 17 for SQL Server'

conn_str = f'mssql+pyodbc://{server}/{database}?trusted_connection=yes&driver={driver}'
engine   = create_engine(conn_str)

# Leer tablas completas
df_customers = pd.read_sql('SELECT * FROM Customers', engine)
df_products  = pd.read_sql('SELECT * FROM Products',  engine)
df_sales     = pd.read_sql('SELECT * FROM Sales',     engine)

print(df_customers.head())
print(df_products.head())
print(df_sales.head())

  customer_id  customer_name   segment        country         city  \
0    AA-10315     Alex Avila  Consumer  United States  Minneapolis   
1    AA-10375   Allen Armold  Consumer  United States         Mesa   
2    AA-10480   Andrew Allen  Consumer  United States      Concord   
3    AA-10645  Anna Andreadi  Consumer  United States      Chester   
4    AB-10015  Aaron Bergman  Consumer  United States      Seattle   

            state  postal_code   region  
0       Minnesota        55407  Central  
1         Arizona        85204     West  
2  North Carolina        28027    South  
3    Pennsylvania        19013     East  
4      Washington        98103     West  
        product_id                                       product_name  \
0  FUR-BO-10000112   Bush Birmingham Collection Bookcase, Dark Cherry   
1  FUR-BO-10000330  Sauder Camden County Barrister Bookcase, Plank...   
2  FUR-BO-10000362                 Sauder Inglewood Library Bookcases   
3  FUR-BO-10000468            O'Sul

In [27]:
query = """
    SELECT 
        s.order_id,
        s.order_date,
        c.customer_name,
        p.product_name,
        s.sales,
        s.profit
    FROM Sales s
    JOIN Customers c ON s.customer_id = c.customer_id
    JOIN Products  p ON s.product_id  = p.product_id
"""

df_reporte = pd.read_sql(query, engine)
df_reporte.head()

,order_id,order_date,customer_name,product_name,sales,profit
0,CA-2016-152156,2016-11-08,Claire Gute,Bush Somerset Collection Bookcase,261.9600,41.9136
1,CA-2016-152156,2016-11-08,Claire Gute,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,219.5820
2,CA-2016-138688,2016-06-12,Darrin Van Huff,Self-Adhesive Address Labels for Typewriters b...,14.6200,6.8714
3,US-2015-108966,2015-10-11,Sean O'Donnell,Bretford CR4500 Series Slim Rectangular Table,957.5775,-383.0310
4,US-2015-108966,2015-10-11,Sean O'Donnell,Eldon Fold 'N Roll Cart System,22.3680,2.5164


In [28]:
print(df_reporte['order_id'].value_counts().head(10))

order_id
CA-2017-100111    14
CA-2017-157987    12
CA-2016-165330    11
US-2016-108504    11
CA-2015-131338    10
CA-2016-105732    10
US-2015-126977    10
US-2016-114013     9
CA-2014-106439     9
CA-2016-145177     9
Name: count, dtype: int64
